In [ ]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


In [ ]:
csv_file_path = f"{path}/twitter_training.csv"
df = pd.read_csv(csv_file_path, header=None)
df.columns = ['ID', 'Source', 'Sentiment', 'Text']
display(df.head())

,ID,Source,Sentiment,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [ ]:
print(df['Sentiment'].value_counts())
print(df['Sentiment'].unique())

Sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64
['Positive' 'Neutral' 'Negative' 'Irrelevant']


In [ ]:
import os

# List the contents of the directory to find the CSV file
dataset_files = os.listdir(path)
print("Files in the dataset directory:", dataset_files)

Files in the dataset directory: ['twitter_validation.csv', 'twitter_training.csv']


In [ ]:
import shutil
import os

# Source directory (where kagglehub downloaded the files)
source_dir = path # 'path' variable already holds this value

# Destination directory (Colab local directory)
dest_dir = '.' # Current working directory

# Define the files to copy
files_to_copy = ['twitter_training.csv', 'twitter_validation.csv']

for filename in files_to_copy:
    source_file_path = os.path.join(source_dir, filename)
    destination_file_path = os.path.join(dest_dir, filename)
    shutil.copy(source_file_path, destination_file_path)
    print(f"Copied '{filename}' to '{destination_file_path}'")

print("Files in Colab local directory:")
print(os.listdir(dest_dir))

Copied 'twitter_training.csv' to './twitter_training.csv'
Copied 'twitter_validation.csv' to './twitter_validation.csv'
Files in Colab local directory:
['.config', 'twitter_validation.csv', 'twitter_training.csv', 'sample_data']


In [ ]:
# Filter out 'Irrelevant' sentiments and create a deep copy (now including 'Neutral')
df_filtered = df[df['Sentiment'].isin(['Positive', 'Negative', 'Neutral'])].copy()

# Map sentiments to numerical labels: Positive = 2, Neutral = 1, Negative = 0
df_filtered['Sentiment_Label'] = df_filtered['Sentiment'].map({'Positive': 2, 'Neutral': 1, 'Negative': 0})

# Display the value counts of the new sentiment labels and the head of the filtered DataFrame
print(df_filtered['Sentiment_Label'].value_counts())
display(df_filtered.head())

Sentiment_Label
0    22542
2    20832
1    18318
Name: count, dtype: int64


,ID,Source,Sentiment,Text,Sentiment_Label
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,2
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,2
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,2
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,2
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,2


## Text Preprocessing

Before training any machine learning model on text data, it's crucial to preprocess the text. This involves several steps to clean and standardize the text, making it more suitable for analysis.

Key preprocessing steps often include:

1.  **Lowercasing**: Converting all text to lowercase to treat words like "The" and "the" as the same.
2.  **Removing Punctuation**: Eliminating punctuation marks (e.g., periods, commas, exclamation points) as they usually don't carry significant semantic meaning for sentiment.
3.  **Removing Stopwords**: Getting rid of common words (e.g., "a", "an", "the", "is") that appear frequently but often don't add much value to the meaning of the text.
4.  **Tokenization**: Breaking down text into individual words or tokens.
5.  **Lemmatization/Stemming**: Reducing words to their base or root form (e.g., "running", "ran", "runs" all become "run"). This helps reduce the vocabulary size and treats variations of a word as a single entity.

Let's implement these steps using NLTK.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize lemmatizer and set of stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str): # Handle non-string input, e.g., None or NaN
        return ""
    text = text.lower() # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers, keep only letters and spaces
    tokens = text.split() # Tokenization
    tokens = [word for word in tokens if word not in stop_words] # Remove stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens] # Lemmatization
    return ' '.join(tokens)

# Apply preprocessing to the 'Text' column
df_filtered['Processed_Text'] = df_filtered['Text'].apply(preprocess_text)

print(df_filtered[['Text', 'Processed_Text']].head())
display(df_filtered.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                                Text  \
0  im getting on borderlands and i will murder yo...   
1  I am coming to the borders and I will kill you...   
2  im getting on borderlands and i will kill you ...   
3  im coming on borderlands and i will murder you...   
4  im getting on borderlands 2 and i will murder ...   

                 Processed_Text  
0  im getting borderland murder  
1            coming border kill  
2    im getting borderland kill  
3   im coming borderland murder  
4  im getting borderland murder  


,ID,Source,Sentiment,Text,Sentiment_Label,Processed_Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,2,im getting borderland murder
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,2,coming border kill
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,2,im getting borderland kill
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,2,im coming borderland murder
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,2,im getting borderland murder


## Feature Extraction: TF-IDF Vectorization

After preprocessing, text data still needs to be converted into a numerical format that machine learning models can understand. One common and effective method for this is TF-IDF (Term Frequency-Inverse Document Frequency).

**TF-IDF** measures how important a word is to a document in a collection or corpus. It's a product of two statistics:

*   **Term Frequency (TF)**: How frequently a term appears in a document.
*   **Inverse Document Frequency (IDF)**: How rare or common a term is across all documents in the corpus. Rare words have higher IDF values.

A high TF-IDF score means a word is very relevant to a particular document, while a low score suggests it's less important or very common across many documents.

We will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to convert our processed text into numerical vectors.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for practicality

# Fit and transform the 'Processed_Text' to get TF-IDF features
X = tfidf_vectorizer.fit_transform(df_filtered['Processed_Text'])
y = df_filtered['Sentiment_Label']

print(f"Shape of TF-IDF features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")

Shape of TF-IDF features (X): (61692, 5000)
Shape of labels (y): (61692,)


## Data Splitting

To properly evaluate our model's performance, we need to split our dataset into training and testing sets. The training set will be used to teach the model, and the testing set will be used to evaluate how well the model generalizes to unseen data.

We'll use `train_test_split` from `sklearn.model_selection` for this purpose.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (49353, 5000)
Shape of X_test: (12339, 5000)
Shape of y_train: (49353,)
Shape of y_test: (12339,)


## Model Training: Logistic Regression

Logistic Regression is a linear model used for binary classification. Despite its name, it's a classification algorithm, not a regression one. It works by estimating the probability of a given input belonging to a certain class.

We will use `LogisticRegression` from `sklearn.linear_model` to train our sentiment classification model.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model for multiclass classification
logistic_model = LogisticRegression(random_state=42, solver='lbfgs', multi_class='auto', max_iter=1000) # 'lbfgs' solver supports multiclass

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression model trained successfully!


## Model Evaluation

After training the model, it's essential to evaluate its performance on the unseen test data. We'll use the following metrics:

*   **Accuracy**: The proportion of correctly classified instances.
*   **Classification Report**: Provides precision, recall, and F1-score for each class, which are crucial for understanding model performance beyond just accuracy, especially in imbalanced datasets.

We will use `accuracy_score` and `classification_report` from `sklearn.metrics`.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = logistic_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7543

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.80      0.79      4509
           1       0.69      0.70      0.70      3664
           2       0.77      0.75      0.76      4166

    accuracy                           0.75     12339
   macro avg       0.75      0.75      0.75     12339
weighted avg       0.75      0.75      0.75     12339



## Sentiment Prediction Function

Now that our model is trained and evaluated, let's create a function that takes a raw text input, preprocesses it, transforms it using the trained TF-IDF vectorizer, and then predicts its sentiment (Positive or Negative) using our Logistic Regression model.

In [ ]:
def predict_sentiment(text):
    # Preprocess the input text
    processed_text = preprocess_text(text)

    # Transform the processed text using the fitted TF-IDF vectorizer
    text_vector = tfidf_vectorizer.transform([processed_text])

    # Predict the sentiment label
    prediction = logistic_model.predict(text_vector)

    # Map the numerical label back to sentiment string
    sentiment_map = {2: 'Positive', 1: 'Neutral', 0: 'Negative'}
    return sentiment_map[prediction[0]]

# Example usage:
sentence1 = "I love this product, it's amazing!"
sentence2 = "This movie was terrible, a complete waste of time."
sentence3 = "The service was okay, nothing special."

print(f"Sentiment of \"'{sentence1}'\": {predict_sentiment(sentence1)}")
print(f"Sentiment of \"'{sentence2}'\": {predict_sentiment(sentence2)}")
print(f"Sentiment of \"'{sentence3}'\": {predict_sentiment(sentence3)}")

Sentiment of "'I love this product, it's amazing!'": Positive
Sentiment of "'This movie was terrible, a complete waste of time.'": Negative
Sentiment of "'The service was okay, nothing special.'": Positive


In [ ]:
new_sentence = "the restaurent was the best "
print(f"Sentiment of \"{new_sentence}\": {predict_sentiment(new_sentence)}")

Sentiment of "the restaurent was the best ": Positive
